In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')


In [ ]:
!pip install SpeechRecognition

In [ ]:
!pip install transformers soundfile


In [ ]:
!pip install --upgrade transformers accelerate

In [ ]:
!git clone https://github.com/VarunGumma/IndicTransTokenizer

In [ ]:
%cd IndicTransTokenizer

In [ ]:
!pip install --editable ./

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM
from IndicTransTokenizer import IndicProcessor, IndicTransTokenizer

tokenizer = IndicTransTokenizer(direction="indic-indic")
ip = IndicProcessor(inference=True)
model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-indic-indic-dist-320M", trust_remote_code=True)

sentences = [
    "கடந்த 4 தேர்தல்களாக தமிழ்நாட்டில் நாம் தமிழர் கட்சி தனித்து போட்டியிட்டு வருகிறது. இந்நிலையில் அந்த கட்சி கடந்த தேர்தலில் கரும்பு விவசாயி சின்னத்தில் போட்டியிட்டடது.",
]

batch = ip.preprocess_batch(sentences, src_lang="tam_Taml", tgt_lang="tel_Telu")
batch = tokenizer(batch, src=True, return_tensors="pt")

with torch.inference_mode():
    outputs = model.generate(**batch, num_beams=5, num_return_sequences=1, max_length=256)

outputs = tokenizer.batch_decode(outputs, src=False)
outputs = ip.postprocess_batch(outputs, lang="tel_Telu")
print(outputs)

In [ ]:
import torch
from transformers import pipeline

# path to the audio file to be transcribed
audio = "/content/H_TA_003_R_M_001_003.wav"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

transcribe = pipeline(task="automatic-speech-recognition", model="vasista22/whisper-tamil-medium", chunk_length_s=30, device=device)
transcribe.model.config.forced_decoder_ids = transcribe.tokenizer.get_decoder_prompt_ids(language="ta", task="transcribe")

print('Transcription: ', transcribe(audio)["text"])


In [ ]:
import librosa as sf
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# Load pre-trained model and processor
processor = AutoProcessor.from_pretrained("vasista22/whisper-tamil-medium")
model = AutoModelForSpeechSeq2Seq.from_pretrained("vasista22/whisper-tamil-medium")

def load_audio(filepath):
  """Load audio from a WAV file"""
  audio, sample_rate = sf.load(filepath)
  return audio, sample_rate

def transcribe(audio, sample_rate):
  # Define a default sample rate if 'sampling_rate' is missing
  expected_sample_rate = getattr(model.config, 'sampling_rate', 16000)  # Assuming default 16kHz

  if sample_rate != expected_sample_rate:
    audio = sf.resample(audio, orig_sr=sample_rate, target_sr=expected_sample_rate)
  inputs = processor(audio, return_tensors="pt")
  outputs = model.generate(**inputs)
  transcription = processor.batch_decode(outputs, skip_special_tokens=True)[0]
  return transcription.strip()

# Example usage
filepath = "/content/H_TA_003_R_M_001_003.wav"  # Replace with your WAV file path
audio, sample_rate = load_audio(filepath)

text = transcribe(audio, sample_rate)

print(f"Transcription: {text}")


In [ ]:
from transformers import VitsModel, AutoTokenizer
import torch

model = VitsModel.from_pretrained("facebook/mms-tts-kff-script_telugu")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-kff-script_telugu")

text = "కాబట్టి గొడుగు పట్టుకుని బయటకు వెళ్లడం మంచిది."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform


In [ ]:
from IPython.display import Audio

Audio(output, rate=model.config.sampling_rate)


In [ ]:
import torch
import speech_recognition as sr
from transformers import Wav2Vec2ForCTC, Wav2Vec2Tokenizer, AutoModelForSeq2SeqLM, VitsModel, AutoTokenizer
from IndicTransTokenizer import IndicProcessor

# Load pre-trained models and tokenizers
audio_tokenizer = Wav2Vec2Tokenizer.from_pretrained("facebook/wav2vec2-large-960h")
audio_model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")

tokenizer = IndicTransTokenizer(direction="indic-indic")
ip = IndicProcessor(inference=True)
text_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-indic-indic-dist-320M", trust_remote_code=True)

speech_model = VitsModel.from_pretrained("facebook/mms-tts-kff-script_telugu")
speech_tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-kff-script_telugu")

# Function to transcribe audio to text
def transcribe_audio(audio_path):
    recognizer = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio_data = recognizer.record(source)

    text = recognizer.recognize_google(audio_data, language="ta-IN")

    return text

# Example usage
audio_path = "/content/drive/MyDrive/SPEECH PROCESSING/audio 1.wav"
tamil_text = transcribe_audio(audio_path)

# Translating Tamil text to Telugu text
batch = ip.preprocess_batch([tamil_text], src_lang="tam_Taml", tgt_lang="tel_Telu")
batch = tokenizer(batch, src=True, return_tensors="pt")

with torch.inference_mode():
    outputs = text_model.generate(**batch, num_beams=5, num_return_sequences=1, max_length=256)

outputs = tokenizer.batch_decode(outputs, src=False)
outputs = ip.postprocess_batch(outputs, lang="tel_Telu")

# Synthesizing Telugu text to Telugu speech
text_input = outputs[0]
inputs = speech_tokenizer(text_input, return_tensors="pt")

with torch.no_grad():
    output = speech_model(**inputs).waveform

from IPython.display import Audio
Audio(output, rate=speech_model.config.sampling_rate)


In [ ]:
import torch
import librosa as sf
from transformers import pipeline, VitsTokenizer, AutoModelForSeq2SeqLM, AutoProcessor, AutoModelForSpeechSeq2Seq, VitsModel, AutoTokenizer
from IndicTransTokenizer import IndicProcessor, IndicTransTokenizer

# Load ASR model
asr_processor = AutoProcessor.from_pretrained("vasista22/whisper-tamil-medium")
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained("vasista22/whisper-tamil-medium")

# Load text-to-text model
txt_tokenizer = IndicTransTokenizer(direction="indic-indic")
txt_processor = IndicProcessor(inference=True)
txt_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-indic-indic-dist-320M", trust_remote_code=True)

# Load text-to-speech model
tts_model = VitsModel.from_pretrained("facebook/mms-tts-kff-script_telugu")
tts_tokenizer = VitsTokenizer.from_pretrained("facebook/mms-tts-kff-script_telugu")

# Define a function for combined inference
def combine_models(input_text):
    # ASR
    audio, sample_rate = sf.load(input_text)
    inputs = asr_processor(audio, return_tensors="pt")
    transcribed_text = asr_model.generate(**inputs)
    transcribed_text = asr_processor.batch_decode(transcribed_text, skip_special_tokens=True)[0]

    # Text-to-Text
    sentences = [transcribed_text]
    batch = txt_processor.preprocess_batch(sentences, src_lang="tam_Taml", tgt_lang="tel_Telu")
    batch = txt_tokenizer(batch, src=True, return_tensors="pt")
    outputs = txt_model.generate(**batch, num_beams=5, num_return_sequences=1, max_length=256)
    translated_text = txt_tokenizer.batch_decode(outputs, src=False)[0]

    # Text-to-Speech
    tts_inputs = tts_tokenizer(translated_text, return_tensors="pt")
    with torch.no_grad():
        audio_output = tts_model(**tts_inputs).waveform

    return audio_output

# Example usage
input_audio_path = "/content/common_voice_ta_19533371.mp3"  # Replace with your audio file path
output_audio = combine_models(input_audio_path)
output_audio.save("output_audio.wav")


In [ ]:
from transformers.modeling_utils import PreTrainedModel
from transformers import BertModel, Wav2Vec2Model, BertTokenizer
import torch.nn as nn
from transformers import pipeline, VitsTokenizer, AutoModelForSeq2SeqLM, AutoProcessor, AutoModelForSpeechSeq2Seq, VitsModel, AutoTokenizer
from IndicTransTokenizer import IndicProcessor, IndicTransTokenizer

class CustomModel(PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        # Load transformers
        self.speech_to_text_model = AutoModelForSpeechSeq2Seq.from_pretrained("vasista22/whisper-tamil-medium")
        self.text_to_text_model = AutoModelForSeq2SeqLM.from_pretrained("ai4bharat/indictrans2-indic-indic-dist-320M", trust_remote_code=True)
        self.text_to_speech_model = VitsModel.from_pretrained("facebook/mms-tts-kff-script_telugu")

        # Tokenizer for text input
        self.tokenizer = IndicTransTokenizer(direction="indic-indic")


        # No need for a classification layer if it's not a classification task
        # Just output the speech from the last model
        self.classifier = None

        # Initialize weights and apply final processing
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, audio_input=None):
        # Forward pass through the models
        speech_output = self.speech_to_text_model(input_values=audio_input).last_hidden_state
        text_output = self.text_to_text_model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        speech_output = self.text_to_speech_model(input_values=text_output).last_hidden_state

        return speech_output



In [ ]:
!pip install datasets

In [ ]:
!pip install sacrebleu

In [ ]:
from transformers import pipeline
from datasets import load_dataset

# Load pre-trained Tamil speech recognition pipeline (whisper-tamil-medium)
pipe = pipeline("automatic-speech-recognition", model="vasista22/whisper-tamil-medium")

def evaluate_model(dataset_name,la, split):
  """
  Evaluates the speech recognition model on a Hugging Face dataset.

  Args:
      dataset_name (str, optional): Name of the Hugging Face dataset to use. Defaults to "common_voice/ta" (Tamil Common Voice).
      split (str, optional): Split of the dataset to use for evaluation (e.g., "test"). Defaults to "test".

  Returns:
      float: Average Word Error Rate (WER) across all audio samples.
  """

  # Load the dataset from Hugging Face
  dataset = load_dataset(dataset_name,la, split=split)

  # Total WER to track across all samples
  total_wer = 0.0

  for idx, datapoint in enumerate(dataset):
    # Extract audio bytes and ground truth text
    audio_bytes = datapoint["audio"]  # Assuming the dataset has "audio" and "text" columns
    ground_truth_text = datapoint["sentence"]

    # Perform speech recognition and calculate WER
    try:
      transcription = pipe(audio_bytes)
      text = transcription[0]['text']
      from sacrebleu import BLEU  # Consider using sacrebleu for WER calculation
      wer = 1 - BLEU(score_fn=bleu_score, n=1).corpus_score([[ground_truth_text]], [[text]])[1]
      total_wer += wer
      print(f"Sample {idx+1} WER: {wer:.4f}")
    except Exception as e:
      print(f"Error during processing sample {idx+1}: {e}")

  # Calculate average WER
  average_wer = total_wer / len(dataset)
  print(f"Average Word Error Rate (WER) across {len(dataset)} samples: {average_wer:.4f}")
  return average_wer

# Example usage
dataset_name = "mozilla-foundation/common_voice_11_0"
la = "ta"  # You can change this to your desired dataset
split = "test[:200]"  # You can change this to another split (e.g., "dev")
average_wer = evaluate_model(dataset_name,la, split)
print(f"Model performance on {dataset_name} ({split}): WER = {average_wer:.4f}")


In [ ]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
)
from IndicTransTokenizer import IndicProcessor


model_name = "ai4bharat/indictrans2-indic-indic-dist-320M"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)

ip = IndicProcessor(inference=True)

input_sentences = [
    "ఈ రోజు వాతావరణం చాలా బాగుంది",
    "చదువు చాలా ముఖ్యం, జ్ఞాపకం ఉంచుకోండి",
    "రేపు సినిమాకి వెళ్ళాలని అనుకుంటున్నాను",
    "మీరు ఎలా ఉన్నారు? నేను బాగున్నాను",
]

src_lang, tgt_lang = "tel_Telu", "tam_Taml"

batch = ip.preprocess_batch(
    input_sentences,
    src_lang=src_lang,
    tgt_lang=tgt_lang,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Tokenize the sentences and generate input encodings
inputs = tokenizer(
    batch,
    truncation=True,
    padding="longest",
    return_tensors="pt",
    return_attention_mask=True,
).to(DEVICE)

# Generate translations using the model
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        use_cache=True,
        min_length=0,
        max_length=256,
        num_beams=5,
        num_return_sequences=1,
    )

# Decode the generated tokens into text
with tokenizer.as_target_tokenizer():
    generated_tokens = tokenizer.batch_decode(
        generated_tokens.detach().cpu().tolist(),
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

# Postprocess the translations, including entity replacement
translations = ip.postprocess_batch(generated_tokens, lang=tgt_lang)

for input_sentence, translation in zip(input_sentences, translations):
    print(f"{src_lang}: {input_sentence}")
    print(f"{tgt_lang}: {translation}")


## **EVALUATION ASR:**

In [ ]:
!pip install datasets

In [ ]:
!pip install evaluate

In [ ]:
!pip install jiwer

In [ ]:
from transformers import pipeline
from datasets import load_dataset
from tqdm import tqdm
from transformers.pipelines.pt_utils import KeyDataset

import torch

if torch.cuda.is_available():
    device = "cuda:0"
    torch_dtype = torch.float16
else:
    device = "cpu"
    torch_dtype = torch.float32

pipe = pipeline("automatic-speech-recognition",
                model="vasista22/whisper-tamil-medium",
                torch_dtype=torch_dtype,
                device=device,
                )

common_voice_test = load_dataset(
    "mozilla-foundation/common_voice_11_0", "ta", split="test[:200]"
)

all_predictions = []

# Run streamed inference
for prediction in tqdm(
    pipe(
        KeyDataset(common_voice_test, "audio"),
        max_new_tokens=128,
        batch_size=32,
    ),
    total=len(common_voice_test),
):
    all_predictions.append(prediction["text"])


In [ ]:
from evaluate import load

wer_metric = load("wer")

wer_ortho = 100 * wer_metric.compute(
    references=common_voice_test["sentence"], predictions=all_predictions
)
wer_ortho